# Phase diagrams with calphy

CuNi system with EAM potential: B. Onat, and S. Durukanoğlu (2013), "An optimized interatomic potential for Cu–Ni alloys with the embedded-atom method", Journal of Physics: Condensed Matter 26(3), 035404. DOI: 10.1088/0953-8984/26/3/035404.

<img src="CuNi.jpg" alt="plot" width="400">

## Melting temperature of pure phases

```
calphy_kernel -i Cu100Ni000.yml -k 0 -s true
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
temp, fe = np.loadtxt('ts-cu.fcc.data-solid-1200-0/temperature_sweep.dat', unpack=True, usecols=(0, 1))

In [ ]:
plt.plot(temp, fe, label='Cu FCC')
plt.xlabel('T (K)')
plt.ylabel('F (eV/atom)')
plt.legend(frameon=False);

In [ ]:
temp_sol, fe_sol = np.loadtxt('ts-cu.fcc.data-solid-1200-0/temperature_sweep.dat', unpack=True, usecols=(0, 1))
temp_lqd, fe_lqd = np.loadtxt('ts-cu.lqd.data-liquid-1200-0/temperature_sweep.dat', unpack=True, usecols=(0, 1))

In [ ]:
plt.plot(temp_sol, fe_sol, label='Cu FCC')
plt.plot(temp_lqd, fe_lqd, label='Cu lqd')
plt.xlabel('T (K)')
plt.ylabel('F (eV/atom)')
plt.legend(frameon=False);

In [ ]:
arg = np.argsort(np.abs(fe_sol-fe_lqd))[0]
plt.plot(temp_sol, fe_sol, label='Cu FCC')
plt.plot(temp_lqd, fe_lqd, label='Cu lqd')
plt.axvline(temp_sol[arg], ls='dashed', c='black', label=f'{np.ceil(temp_sol[arg])} K')
plt.xlabel('T (K)')
plt.ylabel('F (eV/atom)')
plt.legend(frameon=False);

In [ ]:
temp_sol_Ni, fe_sol_Ni = np.loadtxt('ts-ni.fcc.data-solid-1400-0/temperature_sweep.dat', unpack=True, usecols=(0, 1))
temp_lqd_Ni, fe_lqd_Ni = np.loadtxt('ts-ni.lqd.data-liquid-1400-0/temperature_sweep.dat', unpack=True, usecols=(0, 1))

In [ ]:
arg = np.argsort(np.abs(fe_sol_Ni-fe_lqd_Ni))[0]
plt.plot(temp_sol_Ni, fe_sol_Ni, label='Ni FCC')
plt.plot(temp_lqd_Ni, fe_lqd_Ni, label='Ni lqd')
plt.axvline(temp_sol_Ni[arg], ls='dashed', c='black', label=f'{np.ceil(temp_sol_Ni[arg])} K')
plt.xlabel('T (K)')
plt.ylabel('F (eV/atom)')
plt.legend(frameon=False);

## Free energy of liquid with composition

In [ ]:
comps = np.arange(0, 1.25, 0.25)
comps

In [ ]:
from calphy.postprocessing import read_report
from calphy.integrators import kb

In [ ]:
fe_liquid = []

In [ ]:
fe_liquid.append(fe_lqd[-1])

In [ ]:
read_report('fe-cu075ni025.lqd.data-liquid-1400-0')

In [ ]:
read_report('fe-cu075ni025.lqd.data-liquid-1400-0')['results']['free_energy']

In [ ]:
fe_liquid.append(read_report('fe-cu075ni025.lqd.data-liquid-1400-0')['results']['free_energy'])

In [ ]:
fe_liquid.append(read_report('fe-cu050ni050.lqd.data-liquid-1400-0')['results']['free_energy'])
fe_liquid.append(read_report('fe-cu025ni075.lqd.data-liquid-1400-0')['results']['free_energy'])

In [ ]:
fe_liquid.append(fe_lqd_Ni[0])

In [ ]:
plt.plot(comps, fe_liquid, 'o-')
plt.xlabel('x (at% Ni)')
plt.ylabel('F (eV/atom)')
plt.legend(frameon=False);

## Free energy of solid with composition

In [ ]:
fe_solid = []

In [ ]:
fe_solid.append(fe_sol[-1])

In [ ]:
read_report('Cu075-composition_scaling-cu.fcc.data--1400-0')['results']['free_energy']

In [ ]:
fe_solid.append(read_report('Cu075-composition_scaling-cu.fcc.data--1400-0')['results']['free_energy'] + fe_sol[-1] + kb*1400*(0.25*np.log(0.25)+0.75*np.log(0.75)))
fe_solid.append(read_report('Cu050-composition_scaling-cu.fcc.data--1400-0')['results']['free_energy'] + fe_sol[-1] + kb*1400*(0.50*np.log(0.50)+0.50*np.log(0.50)))
fe_solid.append(read_report('Cu025-composition_scaling-cu.fcc.data--1400-0')['results']['free_energy'] + fe_sol[-1] + kb*1400*(0.75*np.log(0.75)+0.25*np.log(0.25)))

In [ ]:
fe_solid.append(fe_sol_Ni[0])

In [ ]:
plt.plot(comps, fe_liquid, 'o-', label='liquid')
plt.plot(comps, fe_solid, 'o-', label='solid')

plt.xlabel('x (at% Ni)')
plt.ylabel('F (eV/atom)')
plt.legend(frameon=False);

## Calculating coexistence

$$
\begin{aligned}
\text{Chemical potential equality:} \quad 
& \mu^{(s)}(x_s^*, T) = \mu^{(l)}(x_l^*, T) = \mu^*(T) \\[6pt]
\text{Grand potential equality:} \quad 
& F^{(s)}(x_s^*, T) - \mu^*(T) x_s^* = F^{(l)}(x_l^*, T) - \mu^*(T) x_l^* = b^*(T)
\end{aligned}
$$

In [ ]:
import numpy as np
from scipy.interpolate import UnivariateSpline
from scipy.optimize import fsolve

In [ ]:
sol_fit = UnivariateSpline(comps, fe_solid, s=0)
lqd_fit = UnivariateSpline(comps, fe_liquid, s=0)
dsol_fit = sol_fit.derivative()
dlqd_fit = lqd_fit.derivative()

In [ ]:
def equations(vars):
    x1, x2 = vars
    mu1, mu2 = dsol_fit(x1), dlqd_fit(x2)
    eq1 = mu1 - mu2
    eq2 = sol_fit(x1) - mu1*x1 - (lqd_fit(x2) - mu2*x2)
    return (eq1, eq2)

In [ ]:
x1_eq, x2_eq = fsolve(equations, (0.0, 1.0))
mu_eq = dsol_fit(x1_eq)

In [ ]:
comp_grid = np.linspace(0, 1, 1000)

In [ ]:
mu_sol = dsol_fit(comp_grid)
mu_lqd = dlqd_fit(comp_grid)

In [ ]:
phi_sol = sol_fit(comp_grid) - mu_sol*comp_grid
phi_lqd = lqd_fit(comp_grid) - mu_lqd*comp_grid

In [ ]:
plt.plot(mu_sol, phi_sol, label='solid')
plt.plot(mu_lqd, phi_lqd, label='liquid')
plt.axvline(mu_eq, ls='dashed', c='black')
plt.xlabel(r'$\mu$ (eV/atom)')
plt.ylabel(r'$\phi$ (eV/atom)')
plt.legend(frameon=False);

In [ ]:
plt.plot(comp_grid, sol_fit(comp_grid), label='solid')
plt.plot(comp_grid, lqd_fit(comp_grid), label='liquid')
plt.plot([x1_eq, x2_eq], [sol_fit(x1_eq), lqd_fit(x2_eq)], c='black')
plt.xlabel(r'$x$ (at% Ni)')
plt.ylabel(r'$F$ (eV/atom)')
plt.legend(frameon=False);

In [ ]:
lhs = min(sol_fit(comp_grid)[0], lqd_fit(comp_grid)[0])
rhs = min(sol_fit(comp_grid)[-1], lqd_fit(comp_grid)[-1])

In [ ]:
sol_mix = sol_fit(comp_grid)-(comp_grid*rhs + comp_grid[::-1]*lhs)
lqd_mix = lqd_fit(comp_grid)-(comp_grid*rhs + comp_grid[::-1]*lhs)

sol_mix_x1 = sol_fit(x1_eq)-(x1_eq*rhs + (1-x1_eq)*lhs)
lqd_mix_x1 = lqd_fit(x2_eq)-(x2_eq*rhs + (1-x2_eq)*lhs)

In [ ]:
x1_eq, x2_eq

In [ ]:
plt.plot(comp_grid, sol_mix, label='solid')
plt.plot(comp_grid, lqd_mix, label='liquid')
plt.plot([x1_eq, x2_eq], [sol_mix_x1, lqd_mix_x1], c='black')
plt.axvline(x1_eq, ls='dashed', c='black')
plt.axvline(x2_eq, ls='dashed', c='black')
plt.xlabel(r'$x$ (at% Ni)')
plt.ylabel(r'$\Delta F$ (eV/atom)')
plt.legend(frameon=False);